# 🚀 Python Performance Engineering — Complete Reference Notebook

> **Your go-to reference for profiling, optimizing, and scaling Python code — from basics to ML pipelines.**

## How to Use This Notebook
- Each section is self-contained. Jump directly to what you need.
- Every concept has a **benchmark comparison** so you see the actual speedup.
- Run cells top-to-bottom for first-time learning; use the TOC for revision.

---

## 📚 Table of Contents

### Part 1 — General Python Performance
1. [Profiling Tools](#1-profiling-tools)
2. [Timing & Benchmarking](#2-timing--benchmarking)
3. [Algorithm Complexity & Data Structures](#3-algorithm-complexity--data-structures)
4. [Built-in Functions & Comprehensions](#4-built-ins--comprehensions)
5. [Memory Management & Profiling](#5-memory-management)
6. [Generators & Lazy Evaluation](#6-generators--lazy-evaluation)
7. [Caching Strategies](#7-caching-strategies)
8. [Concurrency — Threading vs Multiprocessing](#8-concurrency)
9. [Async I/O with asyncio](#9-async-io)
10. [The GIL — What It Is & Working Around It](#10-the-gil)

### Part 2 — NumPy / Pandas Optimization
11. [NumPy Vectorization & Broadcasting](#11-numpy-vectorization)
12. [Pandas — Efficient Operations](#12-pandas-optimization)

### Part 3 — Compiled Extensions & JIT
13. [Numba — JIT Compilation](#13-numba)
14. [Cython Basics](#14-cython)

### Part 4 — ML / Data Engineering
15. [Data Pipeline Optimization](#15-data-pipelines)
16. [Training Optimization (PyTorch)](#16-training-optimization)
17. [Inference Optimization](#17-inference-optimization)

---


---
## 1. Profiling Tools

> **Rule #1: Never optimize without profiling first. Intuition is usually wrong.**

Profiling tells you *where* your program spends its time. The three main tools:
| Tool | Use Case |
|------|----------|
| `cProfile` | Function-level profiling (call counts, cumulative time) |
| `line_profiler` | Line-by-line profiling inside a function |
| `timeit` | Micro-benchmarks for small snippets |


In [ ]:
import cProfile
import pstats
import io

# --- A realistic example: finding slow code ---

def slow_function():
    """Simulate a function with mixed operations."""
    # Expensive: building list with loop
    result = []
    for i in range(10_000):
        result.append(i ** 2)
    
    # Somewhat expensive: sorting
    result.sort(reverse=True)
    
    # Fast: slicing
    top_10 = result[:10]
    return top_10

def fast_function():
    """Optimized equivalent."""
    result = [i ** 2 for i in range(10_000)]
    result.sort(reverse=True)
    return result[:10]


# --- Profile with cProfile ---
pr = cProfile.Profile()
pr.enable()

for _ in range(100):
    slow_function()

pr.disable()

# --- Pretty-print stats ---
stream = io.StringIO()
ps = pstats.Stats(pr, stream=stream).sort_stats('cumulative')
ps.print_stats(10)  # top 10 functions
print(stream.getvalue())


In [ ]:
# --- %timeit equivalent in plain Python (no magic commands needed) ---
import timeit

slow_time = timeit.timeit(slow_function, number=1000)
fast_time = timeit.timeit(fast_function, number=1000)

print(f"slow_function : {slow_time:.4f}s for 1000 runs")
print(f"fast_function : {fast_time:.4f}s for 1000 runs")
print(f"Speedup       : {slow_time / fast_time:.2f}x")


In [ ]:
# --- line_profiler (install if needed) ---
# pip install line_profiler

# In a Jupyter notebook you'd use:
#   %load_ext line_profiler
#   %lprun -f slow_function slow_function()
#
# In plain Python, use the decorator approach:

try:
    from line_profiler import LineProfiler

    lp = LineProfiler()
    lp_wrapper = lp(slow_function)
    lp_wrapper()
    lp.print_stats()
except ImportError:
    print("Install line_profiler: pip install line_profiler")
    print("Then use %lprun -f your_func your_func() in Jupyter")


---
## 2. Timing & Benchmarking

Common pitfalls when measuring performance:
- **Cache effects** — first run is slower (cold cache). Always warm up or use `timeit`.
- **GC interference** — garbage collector can fire mid-benchmark. Disable it for micro-benchmarks.
- **System noise** — run multiple times and take the **minimum** (not mean).


In [ ]:
import timeit
import gc
import time
from contextlib import contextmanager

# ── 1. Context manager for quick timing ──────────────────────
@contextmanager
def timer(label=""):
    start = time.perf_counter()
    yield
    end = time.perf_counter()
    print(f"{label:<30} {(end - start) * 1000:.3f} ms")

# ── 2. Proper micro-benchmark helper ─────────────────────────
def benchmark(fn, n=10_000, repeat=5, label=None):
    """
    Run fn() n times, repeat times. Return min time per call (µs).
    Disables GC for cleaner measurements.
    """
    gc.disable()
    times = []
    for _ in range(repeat):
        t = timeit.timeit(fn, number=n)
        times.append(t)
    gc.enable()
    
    best = min(times) / n * 1_000_000  # µs per call
    label = label or fn.__name__
    print(f"{label:<35} {best:.3f} µs/call")
    return best


# ── 3. Compare: string concatenation approaches ───────────────
def concat_plus(n=1000):
    s = ""
    for i in range(n):
        s += str(i)
    return s

def concat_join(n=1000):
    return "".join(str(i) for i in range(n))

def concat_list_join(n=1000):
    parts = []
    for i in range(n):
        parts.append(str(i))
    return "".join(parts)

print("String concatenation benchmark (1000 elements):")
print("-" * 55)
t1 = benchmark(concat_plus,      label="str += str (loop)")
t2 = benchmark(concat_join,      label="''.join(generator)")
t3 = benchmark(concat_list_join, label="list + ''.join")

print(f"\n''.join is {t1/t2:.1f}x faster than += loop")


In [ ]:
# ── 4. Warm-up matters for JIT-compiled code ─────────────────
import math

def compute_heavy():
    return sum(math.sqrt(i) for i in range(10_000))

# Cold run (first call)
start = time.perf_counter()
compute_heavy()
cold = time.perf_counter() - start

# Warm run (subsequent calls — CPU branch predictor, caches warm)
start = time.perf_counter()
for _ in range(10):
    compute_heavy()
warm = (time.perf_counter() - start) / 10

print(f"Cold run  : {cold*1000:.3f} ms")
print(f"Warm run  : {warm*1000:.3f} ms")
print(f"Ratio     : {cold/warm:.2f}x  ← always warm up before benchmarking!")


---
## 3. Algorithm Complexity & Data Structures

The biggest performance wins come from choosing the **right data structure**, not micro-optimizations.

| Operation | list | dict/set | deque | heapq |
|-----------|------|----------|-------|-------|
| Lookup    | O(n) | **O(1)** | O(n)  | O(n)  |
| Insert front | O(n) | —    | **O(1)** | — |
| Min/Max   | O(n) | —        | O(n)  | **O(log n)** |
| Membership | O(n) | **O(1)** | O(n) | O(n) |


In [ ]:
import random
import time
from collections import deque
import heapq

N = 100_000
data = list(range(N))
random.shuffle(data)

# ── 1. Membership test: list vs set ──────────────────────────
data_list = data[:]
data_set  = set(data)
targets   = random.sample(data, 1000)

with timer("List membership (1000 lookups)"):
    found = [x for x in targets if x in data_list]

with timer("Set  membership (1000 lookups)"):
    found = [x for x in targets if x in data_set]


In [ ]:
# ── 2. Front insertions: list vs deque ───────────────────────
n = 10_000

with timer("list.insert(0, x) × 10k"):
    lst = []
    for i in range(n):
        lst.insert(0, i)   # O(n) each — shifts entire list

with timer("deque.appendleft(x) × 10k"):
    dq = deque()
    for i in range(n):
        dq.appendleft(i)   # O(1) always


In [ ]:
# ── 3. Top-K problem: sort vs heapq ──────────────────────────
import heapq

big_list = random.sample(range(1_000_000), 500_000)
K = 10

with timer("sorted()[-K:] (full sort)"):
    top_k_sort = sorted(big_list)[-K:]

with timer("heapq.nlargest(K, ...)"):
    top_k_heap = heapq.nlargest(K, big_list)

print(f"\nResults match: {sorted(top_k_sort) == sorted(top_k_heap)}")
print("heapq.nlargest is O(n log K) vs O(n log n) for full sort")


In [ ]:
# ── 4. Counting: dict vs Counter vs defaultdict ───────────────
from collections import Counter, defaultdict
import random, string

words = [''.join(random.choices(string.ascii_lowercase, k=5)) for _ in range(100_000)]

def count_dict(words):
    counts = {}
    for w in words:
        counts[w] = counts.get(w, 0) + 1
    return counts

def count_defaultdict(words):
    counts = defaultdict(int)
    for w in words:
        counts[w] += 1
    return counts

def count_counter(words):
    return Counter(words)

t1 = benchmark(lambda: count_dict(words),        label="dict.get(k, 0)")
t2 = benchmark(lambda: count_defaultdict(words), label="defaultdict(int)")
t3 = benchmark(lambda: count_counter(words),     label="Counter (C impl)")
print(f"\nCounter is {t1/t3:.1f}x faster than manual dict counting")


---
## 4. Built-ins & Comprehensions

Python built-in functions are implemented in C — always prefer them over Python loops.


In [ ]:
# ── 1. Loop vs list comprehension vs map ──────────────────────

data = list(range(100_000))

def with_loop(data):
    result = []
    for x in data:
        result.append(x * x)
    return result

def with_comprehension(data):
    return [x * x for x in data]

def with_map(data):
    return list(map(lambda x: x * x, data))

def with_map_no_lambda(data):
    # map with a built-in (no overhead of Python lambda call)
    # For squaring, use pow or operator
    import operator
    return list(map(lambda x: x**2, data))

print("Squaring 100,000 elements:")
print("-" * 45)
t1 = benchmark(lambda: with_loop(data),            label="for loop + .append()")
t2 = benchmark(lambda: with_comprehension(data),   label="list comprehension")
t3 = benchmark(lambda: with_map(data),             label="map + lambda")
print(f"\nComprehension is {t1/t2:.1f}x faster than loop")


In [ ]:
# ── 2. Built-in functions: sum, min, max, any, all ────────────

import random
nums = [random.random() for _ in range(100_000)]

# sum
def manual_sum(data):
    total = 0.0
    for x in data:
        total += x
    return total

print("Summing 100,000 floats:")
t1 = benchmark(lambda: manual_sum(nums), label="manual loop sum")
t2 = benchmark(lambda: sum(nums),        label="built-in sum()")
print(f"sum() is {t1/t2:.1f}x faster\n")

# any / all — short-circuit!
big_list = [False] * 1_000_000 + [True]

def manual_any(lst):
    for x in lst:
        if x: return True
    return False

print("any() on list with True at end (1M elements):")
t1 = benchmark(lambda: manual_any(big_list), label="manual any loop", n=100)
t2 = benchmark(lambda: any(big_list),        label="built-in any()",  n=100)
print(f"any() is {t1/t2:.1f}x faster")


In [ ]:
# ── 3. Dictionary comprehensions & set operations ─────────────

import random
a = set(random.sample(range(10_000), 5000))
b = set(random.sample(range(10_000), 5000))

# Intersection: loop vs set operator
def loop_intersect(a, b):
    return {x for x in a if x in b}

print("Set intersection (5000 elements each):")
t1 = benchmark(lambda: loop_intersect(a, b), label="comprehension over set")
t2 = benchmark(lambda: a & b,                label="a & b (C-level)")
print(f"Built-in set op is {t1/t2:.1f}x faster")


---
## 5. Memory Management & Profiling

Understanding how Python allocates memory helps you avoid:
- **Memory leaks** (objects not getting garbage collected)
- **Excessive copying** (esp. with large arrays/DataFrames)
- **Reference cycles** (use `gc` module to detect)


In [ ]:
import sys
import gc
from collections import namedtuple
import tracemalloc

# ── 1. Object sizes ───────────────────────────────────────────
print("Size of Python objects:")
print(f"  int(0)         : {sys.getsizeof(0)} bytes")
print(f"  int(1000)      : {sys.getsizeof(1000)} bytes")
print(f"  float          : {sys.getsizeof(1.0)} bytes")
print(f"  empty list     : {sys.getsizeof([])} bytes")
print(f"  list[10 ints]  : {sys.getsizeof(list(range(10)))} bytes")
print(f"  empty dict     : {sys.getsizeof({})} bytes")
print(f"  empty set      : {sys.getsizeof(set())} bytes")
print(f"  empty tuple    : {sys.getsizeof(())} bytes")
print()

# Tuples are more memory efficient than lists for fixed data
lst = list(range(1000))
tpl = tuple(range(1000))
print(f"  list of 1000  : {sys.getsizeof(lst)} bytes")
print(f"  tuple of 1000 : {sys.getsizeof(tpl)} bytes")
print(f"  Saving        : {sys.getsizeof(lst) - sys.getsizeof(tpl)} bytes")


In [ ]:
# ── 2. __slots__ — dramatically reduce per-instance overhead ──

class WithoutSlots:
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

class WithSlots:
    __slots__ = ('x', 'y', 'z')
    def __init__(self, x, y, z):
        self.x = x
        self.y = y
        self.z = z

Point = namedtuple('Point', ['x', 'y', 'z'])

# Create 100,000 instances of each
n = 100_000

tracemalloc.start()
objs_no_slots = [WithoutSlots(i, i+1, i+2) for i in range(n)]
snap1 = tracemalloc.take_snapshot()
del objs_no_slots

objs_slots = [WithSlots(i, i+1, i+2) for i in range(n)]
snap2 = tracemalloc.take_snapshot()
del objs_slots

objs_tuple = [Point(i, i+1, i+2) for i in range(n)]
snap3 = tracemalloc.take_snapshot()
del objs_tuple

tracemalloc.stop()

def total_mem(snap):
    return sum(stat.size for stat in snap.statistics('lineno'))

m1 = total_mem(snap1) / 1024 / 1024
m2 = total_mem(snap2) / 1024 / 1024

print(f"100,000 instances — memory usage:")
print(f"  Without __slots__ : ~{m1:.1f} MB")
print(f"  With __slots__    : ~{m2:.1f} MB")
print(f"  Savings           : {(1 - m2/m1)*100:.0f}%")
print()
print("Rule: Use __slots__ for classes instantiated thousands of times")


In [ ]:
# ── 3. tracemalloc — find where memory is being allocated ──────

tracemalloc.start()

# Simulate a leaky pattern
big_cache = {}
def leaky_function(key):
    # Never evicts! Grows forever.
    big_cache[key] = list(range(1000))

for i in range(500):
    leaky_function(i)

snapshot = tracemalloc.take_snapshot()
top_stats = snapshot.statistics('lineno')

print("Top memory allocations:")
for stat in top_stats[:3]:
    print(stat)

tracemalloc.stop()
print()
print(f"Cache size: {len(big_cache)} entries, ~{sys.getsizeof(big_cache)/1024:.0f} KB dict overhead")
print("Solution: use functools.lru_cache or set a max size")


---
## 6. Generators & Lazy Evaluation

Generators compute values **on demand** — they don't hold the entire result in memory.
Critical for large datasets and streaming pipelines.


In [ ]:
import sys
import itertools
import time

# ── 1. Memory comparison: list vs generator ───────────────────

def squares_list(n):
    return [x * x for x in range(n)]

def squares_gen(n):
    return (x * x for x in range(n))

N = 1_000_000

lst = squares_list(N)
gen = squares_gen(N)

print(f"List of {N:,} squares : {sys.getsizeof(lst) / 1024 / 1024:.1f} MB in memory")
print(f"Generator            : {sys.getsizeof(gen)} bytes (regardless of N!)")


In [ ]:
# ── 2. Generator pipelines — composable & memory efficient ─────

# Simulate a log processing pipeline on a large "file"
import random, string

def fake_log_lines(n=1_000_000):
    """Simulate reading log lines (generator — never all in RAM)."""
    levels = ['INFO', 'WARNING', 'ERROR', 'DEBUG']
    for i in range(n):
        level = random.choice(levels)
        msg = ''.join(random.choices(string.ascii_lowercase, k=20))
        yield f"2024-01-01 {level} {msg}"

def only_errors(lines):
    """Filter: keep only ERROR lines."""
    for line in lines:
        if 'ERROR' in line:
            yield line

def extract_message(lines):
    """Transform: get just the message part."""
    for line in lines:
        parts = line.split()
        yield ' '.join(parts[2:])

def first_n(it, n):
    """Take only first N."""
    for i, item in enumerate(it):
        if i >= n: break
        yield item

# The pipeline — nothing runs until we iterate!
pipeline = first_n(
    extract_message(
        only_errors(
            fake_log_lines(1_000_000)
        )
    ),
    n=5
)

print("First 5 error messages from 1M log lines (zero extra RAM):")
for msg in pipeline:
    print(f"  {msg}")


In [ ]:
# ── 3. itertools — the generator toolkit ─────────────────────
import itertools

# islice — lazy slicing
big_range = range(10**9)  # would need 8GB as a list!
first_5 = list(itertools.islice(big_range, 5))
print(f"First 5 from range(1B): {first_5}")

# chain — concatenate iterables without copying
a = range(5)
b = range(5, 10)
combined = list(itertools.chain(a, b))
print(f"chain(range(5), range(5,10)): {combined}")

# groupby — group consecutive items
data = [('A', 1), ('A', 2), ('B', 3), ('B', 4), ('C', 5)]
for key, group in itertools.groupby(data, key=lambda x: x[0]):
    print(f"  {key}: {list(group)}")

# accumulate — running totals
nums = [1, 2, 3, 4, 5]
running_sum = list(itertools.accumulate(nums))
print(f"Running sum of {nums}: {running_sum}")

# product — cartesian product (no nested loops)
colors = ['red', 'blue']
sizes  = ['S', 'M', 'L']
combos = list(itertools.product(colors, sizes))
print(f"Product: {combos}")


---
## 7. Caching Strategies

Caching trades memory for speed. The right strategy depends on:
- **Function purity** (same inputs → same outputs?) → `lru_cache`
- **Mutability of arguments** → `cache` (Python 3.9+) or manual
- **Distributed/shared cache** → Redis


In [ ]:
import functools
import time

# ── 1. lru_cache — memoize pure functions ─────────────────────

@functools.lru_cache(maxsize=None)  # unbounded cache
def fibonacci_cached(n):
    if n < 2:
        return n
    return fibonacci_cached(n - 1) + fibonacci_cached(n - 2)

def fibonacci_naive(n):
    if n < 2:
        return n
    return fibonacci_naive(n - 1) + fibonacci_naive(n - 2)

# Naive is exponential O(2^n), cached is O(n)
n = 35

start = time.perf_counter()
result_naive = fibonacci_naive(n)
naive_time = time.perf_counter() - start

fibonacci_cached.cache_clear()  # clear for fair test
start = time.perf_counter()
result_cached = fibonacci_cached(n)
cached_time = time.perf_counter() - start

print(f"fibonacci({n}) = {result_cached}")
print(f"Naive   : {naive_time*1000:.1f} ms")
print(f"Cached  : {cached_time*1000:.3f} ms")
print(f"Speedup : {naive_time/cached_time:.0f}x")
print(f"Cache info: {fibonacci_cached.cache_info()}")


In [ ]:
# ── 2. Cache with TTL (time-to-live) — for fresh data ─────────
import time
from functools import wraps

def ttl_cache(maxsize=128, ttl=60):
    """LRU cache with TTL expiry. Great for API calls or DB queries."""
    def decorator(fn):
        cache = {}
        timestamps = {}

        @wraps(fn)
        def wrapper(*args):
            now = time.monotonic()
            if args in cache:
                if now - timestamps[args] < ttl:
                    return cache[args]
            result = fn(*args)
            cache[args] = result
            timestamps[args] = now
            # Evict if over maxsize (simplified)
            if len(cache) > maxsize:
                oldest = min(timestamps, key=timestamps.get)
                del cache[oldest], timestamps[oldest]
            return result

        wrapper.cache_clear = lambda: (cache.clear(), timestamps.clear())
        return wrapper
    return decorator


@ttl_cache(maxsize=100, ttl=5)  # expires after 5 seconds
def fetch_user_data(user_id):
    # Simulate a DB call
    time.sleep(0.1)
    return {"id": user_id, "name": f"User_{user_id}"}

print("First call (cold):")
t0 = time.perf_counter()
data = fetch_user_data(42)
print(f"  {data}  [{(time.perf_counter()-t0)*1000:.0f}ms]")

print("Second call (cached):")
t0 = time.perf_counter()
data = fetch_user_data(42)
print(f"  {data}  [{(time.perf_counter()-t0)*1000:.1f}ms]  ← served from cache")


In [ ]:
# ── 3. Cache busting anti-patterns ───────────────────────────

# WRONG — mutable default args as cache keys won't work
@functools.lru_cache(maxsize=128)
def bad_cache(lst):  # TypeError: unhashable type: 'list'
    pass

# CORRECT — convert mutable args to hashable types
@functools.lru_cache(maxsize=128)
def good_cache(tpl):  # Pass tuple instead of list
    return sum(tpl)

result = good_cache((1, 2, 3, 4, 5))
print(f"good_cache((1,2,3,4,5)) = {result}")

# For dict args, use frozenset of items:
@functools.lru_cache(maxsize=128)
def cache_with_dict(frozen_params):
    params = dict(frozen_params)
    return sum(params.values())

params = frozenset({'a': 1, 'b': 2, 'c': 3}.items())
print(f"cache_with_dict(frozenset) = {cache_with_dict(params)}")


---
## 8. Concurrency — Threading vs Multiprocessing

| | Threading | Multiprocessing |
|-|-----------|----------------|
| Best for | I/O-bound (network, disk) | CPU-bound (computation) |
| Memory | Shared | Separate per process |
| GIL | Blocked for CPU work | Bypasses GIL |
| Overhead | Low | High (process spawn) |


In [ ]:
import threading
import multiprocessing
import time
import math
import urllib.request
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

# ── 1. CPU-bound: multiprocessing wins ───────────────────────

def cpu_heavy(n):
    """Pure CPU work — computing primes."""
    return sum(1 for i in range(2, n) if all(i % d != 0 for d in range(2, int(math.sqrt(i))+1)))

NUMBERS = [50_000] * 4  # 4 tasks

# Sequential
start = time.perf_counter()
sequential = [cpu_heavy(n) for n in NUMBERS]
t_seq = time.perf_counter() - start

# Threading (GIL prevents true parallelism for CPU work)
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=4) as ex:
    threaded = list(ex.map(cpu_heavy, NUMBERS))
t_thread = time.perf_counter() - start

# Multiprocessing (true parallelism)
start = time.perf_counter()
with ProcessPoolExecutor(max_workers=4) as ex:
    parallel = list(ex.map(cpu_heavy, NUMBERS))
t_mp = time.perf_counter() - start

print("CPU-bound task (count primes up to 50,000 × 4 tasks):")
print(f"  Sequential       : {t_seq:.2f}s")
print(f"  ThreadPoolExecutor: {t_thread:.2f}s  ← GIL prevents speedup")
print(f"  ProcessPoolExecutor: {t_mp:.2f}s  ← real parallelism!")
print(f"  Multiprocessing speedup: {t_seq/t_mp:.1f}x")


In [ ]:
import concurrent.futures
import time

# ── 2. I/O-bound: threading wins ─────────────────────────────

def simulate_io_task(task_id, delay=0.2):
    """Simulate a DB query / API call."""
    time.sleep(delay)
    return f"Result-{task_id}"

TASKS = list(range(10))  # 10 I/O tasks, each 200ms

# Sequential — 2 seconds total
start = time.perf_counter()
sequential = [simulate_io_task(i) for i in TASKS]
t_seq = time.perf_counter() - start

# Threading — concurrent I/O
start = time.perf_counter()
with ThreadPoolExecutor(max_workers=10) as ex:
    threaded = list(ex.map(simulate_io_task, TASKS))
t_thread = time.perf_counter() - start

print("I/O-bound task (10 tasks × 200ms each):")
print(f"  Sequential  : {t_seq:.2f}s")
print(f"  10 Threads  : {t_thread:.2f}s  ← near perfect scaling!")
print(f"  Speedup     : {t_seq/t_thread:.1f}x")
print()
print("Key insight: threads release GIL during I/O — real concurrency!")


---
## 9. Async I/O with asyncio

`asyncio` is single-threaded concurrency. The event loop switches between tasks at `await` points.
- **Better than threads** for high-concurrency I/O (thousands of connections)
- **Not better** for CPU-bound work (still single thread)


In [ ]:
import asyncio
import time
import random

# ── 1. Basic async vs sync ────────────────────────────────────

async def fetch_data_async(url_id, delay):
    """Simulates an async HTTP request."""
    await asyncio.sleep(delay)  # yields control to event loop
    return f"data_from_{url_id}"

async def fetch_all_async(tasks):
    """Run all fetches concurrently."""
    results = await asyncio.gather(*[fetch_data_async(i, d) for i, d in tasks])
    return results

# Simulate 10 requests with random delays
tasks = [(i, random.uniform(0.1, 0.5)) for i in range(10)]
total_delay = sum(d for _, d in tasks)

# Sequential would take sum of all delays
# Async takes ~max of all delays

start = time.perf_counter()
results = asyncio.run(fetch_all_async(tasks))
elapsed = time.perf_counter() - start

print(f"10 async tasks (total delay if sequential: {total_delay:.1f}s)")
print(f"asyncio.gather() completed in: {elapsed:.2f}s")
print(f"Speedup: {total_delay/elapsed:.1f}x")
print(f"Results: {results[:3]}...")


In [ ]:
# ── 2. Producer-Consumer with asyncio.Queue ──────────────────

import asyncio

async def producer(queue, n_items):
    for i in range(n_items):
        await asyncio.sleep(0.01)  # simulate producing work
        await queue.put(f"item_{i}")
        print(f"  Produced item_{i}")
    await queue.put(None)  # sentinel

async def consumer(queue, consumer_id):
    while True:
        item = await queue.get()
        if item is None:
            await queue.put(None)  # pass sentinel along
            break
        await asyncio.sleep(0.02)  # simulate processing
        print(f"  Consumer-{consumer_id} processed {item}")
        queue.task_done()

async def main():
    queue = asyncio.Queue(maxsize=3)  # bounded queue — backpressure!
    
    prod = asyncio.create_task(producer(queue, 6))
    cons1 = asyncio.create_task(consumer(queue, 1))
    cons2 = asyncio.create_task(consumer(queue, 2))
    
    await asyncio.gather(prod, cons1, cons2)

print("Async Producer-Consumer pipeline:")
asyncio.run(main())


In [ ]:
# ── 3. asyncio vs threading — when to use which ───────────────

print("""
Decision guide:
─────────────────────────────────────────────────────────
Task type          → Best tool
─────────────────────────────────────────────────────────
Few slow I/O calls → threading.Thread or ThreadPoolExecutor
1000s of connections → asyncio (lower overhead per task)
CPU-heavy work     → multiprocessing.ProcessPoolExecutor
Mixed (web server) → asyncio + ProcessPoolExecutor for CPU
─────────────────────────────────────────────────────────

asyncio overhead per task : ~1-5 µs
threading overhead/thread : ~50-100 µs
process overhead          : ~10-100 ms

For 10,000 concurrent DB queries → asyncio wins clearly.
For 4 CPU cores doing number crunching → multiprocessing.
""")


---
## 10. The GIL — Global Interpreter Lock

The GIL is a mutex in CPython that ensures only **one thread executes Python bytecode at a time**.

**Why does it exist?** Simplifies memory management (reference counting is not thread-safe without it).

**Python 3.13+** introduced experimental no-GIL builds (`--disable-gil`).


In [ ]:
import threading
import time

# ── 1. Demonstrating GIL impact on CPU-bound threading ─────────

counter = 0

def increment_unsafe(n):
    global counter
    for _ in range(n):
        counter += 1  # NOT atomic! Race condition exists

# Run 2 threads — with GIL, this won't crash but also won't parallelize
counter = 0
N = 1_000_000

start = time.perf_counter()
t1 = threading.Thread(target=increment_unsafe, args=(N,))
t2 = threading.Thread(target=increment_unsafe, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()
elapsed = time.perf_counter() - start

print(f"Expected counter : {2*N:,}")
print(f"Actual counter   : {counter:,}")
print(f"  (race condition! += is not atomic even with GIL)")
print(f"Time (2 threads) : {elapsed:.3f}s")

# Sequential
counter = 0
start = time.perf_counter()
increment_unsafe(N)
increment_unsafe(N)
t_seq = time.perf_counter() - start
print(f"Time (sequential): {t_seq:.3f}s")
print(f"Threads were {'faster' if elapsed < t_seq else 'slower or same'} — GIL effect!")


In [ ]:
# ── 2. Thread-safe counting with Lock ────────────────────────
import threading

lock = threading.Lock()
safe_counter = 0

def increment_safe(n):
    global safe_counter
    for _ in range(n):
        with lock:
            safe_counter += 1

safe_counter = 0
N = 100_000  # smaller N — locking is expensive

t1 = threading.Thread(target=increment_safe, args=(N,))
t2 = threading.Thread(target=increment_safe, args=(N,))
t1.start(); t2.start()
t1.join(); t2.join()

print(f"With Lock — Expected: {2*N:,}, Got: {safe_counter:,} ✓")
print()
print("""GIL Workarounds:
  1. multiprocessing    — separate processes, each has own GIL
  2. C extensions       — release GIL in C code (NumPy does this!)
  3. Cython with nogil  — mark sections as GIL-free
  4. Numba @njit        — JIT-compiled, releases GIL
  5. Python 3.13+       — experimental no-GIL mode
""")


---
## 11. NumPy Vectorization & Broadcasting

NumPy operations run in C, releasing the GIL. The core rule:
> **Eliminate Python loops over array elements. Replace with NumPy operations.**


In [ ]:
import numpy as np
import time

# ── 1. Loop vs vectorized ──────────────────────────────────────

N = 1_000_000
a = np.random.rand(N)
b = np.random.rand(N)

# Python loop
def python_dot(a, b):
    total = 0.0
    for x, y in zip(a, b):
        total += x * y
    return total

# NumPy vectorized
def numpy_dot(a, b):
    return np.dot(a, b)

t1 = benchmark(lambda: python_dot(a, b), n=10, label="Python loop dot product")
t2 = benchmark(lambda: numpy_dot(a, b),  n=10, label="np.dot(a, b)")
print(f"NumPy is {t1/t2:.0f}x faster!")


In [ ]:
# ── 2. Broadcasting — avoid tiling arrays ─────────────────────

# Goal: subtract row mean from each row of a 2D matrix

matrix = np.random.rand(1000, 500)

def subtract_mean_loop(m):
    result = np.empty_like(m)
    for i in range(m.shape[0]):
        result[i] = m[i] - m[i].mean()
    return result

def subtract_mean_broadcast(m):
    # m.mean(axis=1) → shape (1000,)
    # Reshape to (1000, 1) to broadcast against (1000, 500)
    return m - m.mean(axis=1, keepdims=True)

# Verify correctness
np.testing.assert_allclose(
    subtract_mean_loop(matrix),
    subtract_mean_broadcast(matrix)
)

t1 = benchmark(lambda: subtract_mean_loop(matrix),      n=100, label="row-wise loop")
t2 = benchmark(lambda: subtract_mean_broadcast(matrix), n=100, label="broadcast")
print(f"Broadcasting is {t1/t2:.1f}x faster")
print()
print("Broadcasting rules:")
print("  (1000, 500) - (1000, 1) → (1000, 500)  ✓")
print("  (1000, 500) - (1000,)   → ERROR (ambiguous axis)")
print("  keepdims=True is the cleanest way to preserve shape for broadcasting")


In [ ]:
# ── 3. Avoid unnecessary copies ───────────────────────────────

arr = np.random.rand(5_000_000)

# These create copies (extra memory + time):
def with_copy():
    return arr.copy() * 2 + 1

# In-place operations (no copy):
def inplace_ops(a):
    result = a * 2         # one copy needed
    result += 1            # in-place! no extra copy
    return result

# Even better: np.multiply with out=
def with_out(a):
    result = np.empty_like(a)
    np.multiply(a, 2, out=result)
    result += 1
    return result

print("Operations on 5M element array:")
t1 = benchmark(lambda: with_copy(),    n=50, label="copy + ops (2 allocs)")
t2 = benchmark(lambda: inplace_ops(arr), n=50, label="1 copy + in-place +=")
t3 = benchmark(lambda: with_out(arr),   n=50, label="np.multiply(out=)")
print(f"\nIn-place is {t1/t2:.1f}x faster than extra copy approach")


In [ ]:
# ── 4. Fancy indexing vs boolean masks ────────────────────────

data = np.random.randn(1_000_000)

# Boolean mask — returns copy
def bool_mask(d):
    return d[d > 0]  # select positives

# np.where — faster for condition + replacement
def where_clip(d):
    return np.where(d > 0, d, 0.0)  # clip negatives to 0

t1 = benchmark(lambda: bool_mask(data),  n=100, label="boolean mask d[d>0]")
t2 = benchmark(lambda: where_clip(data), n=100, label="np.where(d>0, d, 0)")
print(f"np.where is {t1/t2:.1f}x faster for replacement patterns")

# Vectorized clip
t3 = benchmark(lambda: np.clip(data, 0, None), n=100, label="np.clip(data, 0, None)")
print(f"np.clip is {t1/t3:.1f}x faster for clipping")


---
## 12. Pandas — Efficient Operations

Pandas is built on NumPy. The same principle applies: avoid Python loops over rows.


In [ ]:
import pandas as pd
import numpy as np

# Create a realistic DataFrame
N = 500_000
np.random.seed(42)

df = pd.DataFrame({
    'user_id'  : np.random.randint(1, 10_001, N),
    'amount'   : np.random.exponential(100, N).round(2),
    'category' : np.random.choice(['A', 'B', 'C', 'D'], N),
    'date'     : pd.date_range('2023-01-01', periods=N, freq='1min'),
    'score'    : np.random.rand(N),
})

print(f"DataFrame: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(df.dtypes)
print(df.head(3))


In [ ]:
# ── 1. iterrows vs apply vs vectorized ────────────────────────

# Goal: flag rows where amount > 200 and score > 0.8

# TERRIBLE: iterrows (Python loop over rows)
def with_iterrows(df):
    flags = []
    for _, row in df.iterrows():
        flags.append(row['amount'] > 200 and row['score'] > 0.8)
    return flags

# OK: apply (still Python, but cleaner)
def with_apply(df):
    return df.apply(lambda row: row['amount'] > 200 and row['score'] > 0.8, axis=1)

# BEST: vectorized boolean operations
def vectorized(df):
    return (df['amount'] > 200) & (df['score'] > 0.8)

sample = df.head(10_000)  # use sample for iterrows (it's that slow)

t1 = benchmark(lambda: with_iterrows(sample), n=5, label="iterrows (10K rows)")
t2 = benchmark(lambda: with_apply(sample),    n=5, label="apply (10K rows)")
t3 = benchmark(lambda: vectorized(df),        n=100, label="vectorized (500K rows)")

print(f"\nVectorized on 500K rows is {t1/t3*(500_000/10_000):.0f}x faster than iterrows on 10K rows!")
print("Never use iterrows for computation. Use vectorized ops or np.where.")


In [ ]:
# ── 2. groupby optimization ────────────────────────────────────

# Goal: compute mean amount per category

# Good — groupby is highly optimized
def groupby_agg(df):
    return df.groupby('category')['amount'].mean()

# Using transform (for broadcast back to original shape)
def groupby_transform(df):
    return df.groupby('category')['amount'].transform('mean')

t1 = benchmark(lambda: groupby_agg(df),       n=50, label="groupby + agg")
t2 = benchmark(lambda: groupby_transform(df), n=50, label="groupby + transform")

print("Category means:")
print(groupby_agg(df).round(2))


In [ ]:
# ── 3. Categoricals — reduce memory + speed up groupby ─────────

# Convert string categories to Categorical dtype
print("Before categorical conversion:")
print(f"  'category' column memory: {df['category'].memory_usage(deep=True)/1024:.0f} KB")
print(f"  dtype: {df['category'].dtype}")

df['category'] = df['category'].astype('category')

print("\nAfter conversion:")
print(f"  'category' column memory: {df['category'].memory_usage(deep=True)/1024:.0f} KB")
print(f"  dtype: {df['category'].dtype}")

# Now groupby is faster too
t1 = benchmark(lambda: df.groupby('category')['amount'].mean(), n=100, label="groupby on Categorical")
print(f"\nGroupby on Categorical: {t1:.1f} µs/call")


In [ ]:
# ── 4. query() and eval() for large DataFrames ────────────────

# Standard boolean indexing
def standard_filter(df):
    return df[(df['amount'] > 150) & (df['score'] > 0.7)]

# pd.eval — parses expression, avoids intermediate arrays
def eval_filter(df):
    return df.query('amount > 150 and score > 0.7')

t1 = benchmark(lambda: standard_filter(df), n=100, label="boolean indexing")
t2 = benchmark(lambda: eval_filter(df),     n=100, label="df.query()")

print(f"query() is {t1/t2:.1f}x faster on large DataFrames (avoids intermediate arrays)")


---
## 13. Numba — JIT Compilation

Numba compiles Python functions to native machine code using LLVM.
Best for: numerical loops that can't be easily vectorized with NumPy.


In [ ]:
# Note: numba must be installed: pip install numba

try:
    import numba
    from numba import njit, prange
    import numpy as np
    NUMBA_AVAILABLE = True
    print(f"Numba {numba.__version__} available ✓")
except ImportError:
    NUMBA_AVAILABLE = False
    print("Numba not installed. Run: pip install numba")
    print("Showing code structure only.")


In [ ]:
if NUMBA_AVAILABLE:
    import numpy as np
    from numba import njit, prange
    
    # ── 1. Basic @njit ──────────────────────────────────────────
    
    def python_pairwise_dist(X):
        """Pairwise Euclidean distances — O(n²) — hard to vectorize cleanly."""
        n = len(X)
        D = np.zeros((n, n))
        for i in range(n):
            for j in range(i+1, n):
                diff = X[i] - X[j]
                D[i, j] = D[j, i] = np.sqrt(np.dot(diff, diff))
        return D
    
    @njit(parallel=True)
    def numba_pairwise_dist(X):
        """Same algorithm, JIT compiled + parallel loops."""
        n = len(X)
        D = np.zeros((n, n))
        for i in prange(n):  # prange → parallel range
            for j in range(i+1, n):
                diff = X[i] - X[j]
                dist = 0.0
                for k in range(len(diff)):
                    dist += diff[k] ** 2
                D[i, j] = D[j, i] = dist ** 0.5
        return D
    
    X = np.random.rand(300, 10)
    
    # Warm up Numba (first call compiles)
    _ = numba_pairwise_dist(X)
    
    t1 = benchmark(lambda: python_pairwise_dist(X), n=5, label="Python loops")
    t2 = benchmark(lambda: numba_pairwise_dist(X),  n=5, label="@njit(parallel=True)")
    print(f"\nNumba speedup: {t1/t2:.0f}x")

else:
    print("""
    Example: Pairwise distance with Numba
    
    @njit(parallel=True)
    def pairwise_dist(X):
        n = len(X)
        D = np.zeros((n, n))
        for i in prange(n):          # parallel range — uses all CPU cores
            for j in range(i+1, n):
                diff = X[i] - X[j]
                D[i, j] = D[j, i] = np.sqrt(np.dot(diff, diff))
        return D
    """)


In [ ]:
if NUMBA_AVAILABLE:
    from numba import njit
    import numpy as np
    
    # ── 2. Numba with cache=True (avoid recompiling on restart) ──
    
    @njit(cache=True)   # saves compiled code to disk
    def running_statistics(arr):
        """Online mean and variance (Welford's algorithm) — hard to vectorize."""
        n = len(arr)
        mean = 0.0
        M2 = 0.0
        for i in range(n):
            delta = arr[i] - mean
            mean += delta / (i + 1)
            delta2 = arr[i] - mean
            M2 += delta * delta2
        variance = M2 / n if n > 1 else 0.0
        return mean, variance
    
    arr = np.random.rand(1_000_000)
    
    _ = running_statistics(arr)  # warmup
    
    mean, var = running_statistics(arr)
    print(f"Mean     : {mean:.6f}  (expected ~0.5)")
    print(f"Variance : {var:.6f}  (expected ~0.083)")
    
    t = benchmark(lambda: running_statistics(arr), n=100, label="Numba Welford")
    print(f"\nProcessed 1M elements in {t:.1f} µs")

else:
    print("Install numba to run this cell.")


---
## 14. Cython Basics

Cython lets you add C-level type annotations to Python code, then compile it.
Best for: wrapping C libraries, when Numba isn't suitable.


In [ ]:
# Cython requires a build step. Here's the pattern:

cython_example = '''
# file: fast_sum.pyx

def python_sum(lst):
    """Pure Python — slow."""
    total = 0
    for x in lst:
        total += x
    return total

def cython_typed_sum(list lst):
    """Cython with type declaration."""
    cdef double total = 0.0
    cdef int i
    cdef int n = len(lst)
    for i in range(n):
        total += lst[i]
    return total

def cython_array_sum(double[:] arr):
    """Typed memoryview — fastest."""
    cdef double total = 0.0
    cdef int i
    for i in range(len(arr)):
        total += arr[i]
    return total
'''

# setup.py to compile:
setup_example = '''
from setuptools import setup
from Cython.Build import cythonize

setup(
    ext_modules=cythonize("fast_sum.pyx", annotate=True)
)
# Build with: python setup.py build_ext --inplace
'''

print("Cython workflow:")
print("  1. Write .pyx file with type annotations")
print("  2. Compile: python setup.py build_ext --inplace")
print("  3. Import like a normal module: from fast_sum import cython_typed_sum")
print()
print("Key type declarations:")
print("  cdef int i          # C int")
print("  cdef double x       # C double")
print("  cdef double[:] arr  # typed memoryview (numpy array)")
print("  cpdef func(...)     # callable from both Python and C")


---
## 15. Data Pipeline Optimization

For ML training, data loading is often the bottleneck — not the model itself.


In [ ]:
import time
import random
import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor
import numpy as np

# ── 1. Prefetching pattern ────────────────────────────────────

import queue
import threading

class PrefetchDataLoader:
    """
    Loads next batch in background while GPU trains on current batch.
    Eliminates data loading wait time.
    """
    def __init__(self, dataset, batch_size=32, prefetch=2):
        self.dataset    = dataset
        self.batch_size = batch_size
        self.queue      = queue.Queue(maxsize=prefetch)
        self._stop      = threading.Event()
        
        self._thread = threading.Thread(target=self._load_loop, daemon=True)
        self._thread.start()
    
    def _load_batch(self, indices):
        """Simulate loading + augmenting a batch (I/O + light CPU)."""
        time.sleep(0.01)  # disk I/O simulation
        return np.stack([self.dataset[i] for i in indices])
    
    def _load_loop(self):
        n = len(self.dataset)
        indices = list(range(n))
        random.shuffle(indices)
        for start in range(0, n - self.batch_size, self.batch_size):
            if self._stop.is_set(): break
            batch_idx = indices[start:start + self.batch_size]
            batch = self._load_batch(batch_idx)
            self.queue.put(batch)
        self.queue.put(None)  # sentinel
    
    def __iter__(self):
        while True:
            batch = self.queue.get()
            if batch is None: break
            yield batch
    
    def stop(self):
        self._stop.set()


# Simulate a dataset
dataset = [np.random.rand(224, 224, 3).astype(np.float32) for _ in range(200)]

def simulate_training_step(batch):
    """Simulate GPU computation."""
    time.sleep(0.05)  # GPU forward+backward pass

# Without prefetch
start = time.perf_counter()
for i in range(0, 96, 32):
    batch = np.stack(dataset[i:i+32])  # load
    time.sleep(0.01)                    # loading time
    simulate_training_step(batch)       # train
t_no_prefetch = time.perf_counter() - start

# With prefetch
start = time.perf_counter()
loader = PrefetchDataLoader(dataset, batch_size=32)
for batch in loader:
    simulate_training_step(batch)  # load happens in background!
t_prefetch = time.perf_counter() - start

print(f"Without prefetch : {t_no_prefetch:.2f}s")
print(f"With prefetch    : {t_prefetch:.2f}s")
print(f"Speedup          : {t_no_prefetch/t_prefetch:.2f}x")


In [ ]:
# ── 2. Chunked processing — don't load everything at once ──────
import numpy as np

def process_large_dataset_naive(n=10_000_000):
    """Loads everything into RAM first. Bad for large datasets."""
    # This would fail on a 100GB dataset
    all_data = np.random.rand(n)  # pretend this is loaded from disk
    result = np.sqrt(all_data)
    return result.mean()

def process_large_dataset_chunked(n=10_000_000, chunk_size=100_000):
    """Process in chunks — constant memory footprint."""
    total = 0.0
    processed = 0
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        chunk = np.random.rand(end - start)
        total += np.sqrt(chunk).sum()
        processed += (end - start)
    return total / processed

result = process_large_dataset_chunked()
print(f"Chunked mean of sqrt: {result:.6f}  (expected ~{2/3:.6f})")
print()
print("Rule: chunk_size = what fits comfortably in L3 cache (1-10MB typically)")
print(f"      For float32: 1MB = {1024*1024//4:,} elements")


---
## 16. Training Optimization (PyTorch)

Key levers for faster training:
1. Mixed precision (fp16/bf16) — 2x memory reduction, ~1.5-3x speedup on modern GPUs
2. Gradient accumulation — simulate large batch size without large GPU RAM
3. torch.compile — graph-level optimization (PyTorch 2.0+)
4. Proper DataLoader workers


In [ ]:
# ── PyTorch training best practices (code patterns) ──────────

try:
    import torch
    TORCH_AVAILABLE = True
    print(f"PyTorch {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    TORCH_AVAILABLE = False
    print("PyTorch not installed — showing code patterns only.")


In [ ]:
# ── 1. Mixed precision training ───────────────────────────────

mixed_precision_template = '''
import torch
from torch.cuda.amp import autocast, GradScaler

model = MyModel().cuda()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
scaler = GradScaler()  # handles gradient scaling for fp16

for batch in dataloader:
    inputs, labels = batch
    inputs, labels = inputs.cuda(), labels.cuda()
    
    optimizer.zero_grad()
    
    # ── Forward pass in fp16 ──
    with autocast(dtype=torch.float16):   # or bfloat16 for A100/H100
        outputs = model(inputs)
        loss = criterion(outputs, labels)
    
    # ── Backward pass with scaled gradients ──
    scaler.scale(loss).backward()         # scale to avoid fp16 underflow
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    scaler.step(optimizer)
    scaler.update()

# bfloat16 (bf16) — preferred on Ampere+ GPUs (A100, H100, 4090):
#   - Same dynamic range as fp32
#   - No gradient scaler needed!
# with autocast(dtype=torch.bfloat16):
#     ...
'''

print("Mixed Precision Training Pattern:")
print(mixed_precision_template)


In [ ]:
# ── 2. Gradient accumulation ─────────────────────────────────

gradient_accumulation_template = '''
# Simulate batch_size=256 with only 32 samples per step
ACCUMULATION_STEPS = 8  # 8 × 32 = 256 effective batch size
BATCH_SIZE = 32         # what fits in GPU RAM

model.train()
optimizer.zero_grad()

for step, (inputs, labels) in enumerate(dataloader):
    with autocast():
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss = loss / ACCUMULATION_STEPS  # normalize

    scaler.scale(loss).backward()

    if (step + 1) % ACCUMULATION_STEPS == 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
'''

print("Gradient Accumulation Pattern:")
print(gradient_accumulation_template)


In [ ]:
# ── 3. DataLoader configuration ───────────────────────────────

dataloader_template = '''
import torch
from torch.utils.data import DataLoader

# WRONG — single worker, no pinning
slow_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=0,      # no parallelism
    pin_memory=False,   # no pinned memory
)

# RIGHT — parallel workers, pinned memory, prefetch
fast_loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,              # = num CPU cores / 2 typically
    pin_memory=True,            # page-locked RAM → faster GPU transfer
    prefetch_factor=2,          # prefetch 2 batches per worker
    persistent_workers=True,    # don't restart workers each epoch
    drop_last=True,             # consistent batch size (avoid recompile)
)

# Rule of thumb for num_workers:
#   - Start with num_workers = 4
#   - Increase until GPU utilization stays >90%
#   - Too many → diminishing returns + RAM pressure
'''

print("DataLoader Optimization:")
print(dataloader_template)


---
## 17. Inference Optimization

Training is done once; inference runs millions of times. Every ms matters.


In [ ]:
# ── 1. torch.compile (PyTorch 2.0+) ──────────────────────────

compile_template = '''
import torch

model = MyModel().eval().cuda()

# PyTorch 2.0+ — compile to optimized kernel
# First call is slow (compilation), subsequent calls are fast
compiled_model = torch.compile(model, mode="reduce-overhead")
# modes: "default", "reduce-overhead" (loops), "max-autotune" (aggressive)

# Inference loop
with torch.no_grad():
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        output = compiled_model(input_tensor)
'''

print("torch.compile pattern:")
print(compile_template)

# ── 2. Quantization ───────────────────────────────────────────
quantization_template = '''
# Post-Training Quantization (PTQ) — no retraining needed
import torch.quantization

model_fp32 = MyModel().eval()

# Dynamic quantization — weights int8, activations float
model_int8 = torch.quantization.quantize_dynamic(
    model_fp32,
    {torch.nn.Linear},     # which layers to quantize
    dtype=torch.qint8
)
# Result: ~4x smaller model, ~2-3x faster on CPU inference

# Static quantization (better, needs calibration data)
model_fp32.qconfig = torch.quantization.get_default_qconfig("fbgemm")
torch.quantization.prepare(model_fp32, inplace=True)

# Calibrate with representative data
for sample in calibration_loader:
    model_fp32(sample)

torch.quantization.convert(model_fp32, inplace=True)
# Now fully quantized int8 model
'''

print("\nQuantization pattern:")
print(quantization_template)


In [ ]:
# ── 3. Batching inference requests ────────────────────────────

batching_template = '''
# WRONG — process one request at a time
def serve_one_at_a_time(requests):
    results = []
    for req in requests:
        tensor = preprocess(req)           # shape: (1, ...)
        with torch.no_grad():
            out = model(tensor.cuda())
        results.append(postprocess(out))
    return results                         # GPU is underutilized!

# RIGHT — batch requests
def serve_batched(requests, batch_size=32):
    results = []
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i+batch_size]
        tensors = torch.stack([preprocess(r) for r in batch])  # (B, ...)
        with torch.no_grad():
            outs = model(tensors.cuda())                        # one GPU call
        results.extend([postprocess(o) for o in outs])
    return results
'''

print("Batched inference pattern:")
print(batching_template)


---
## 🏁 Summary: Performance Optimization Decision Tree

```
Is it slow?
├── Profile first! (cProfile, line_profiler)
│
├── Is it algorithmic? (O(n²) when O(n) exists)
│   └── Fix the algorithm. Data structure choice. No code trick beats this.
│
├── Is it I/O bound? (network, disk)
│   ├── Few tasks  → ThreadPoolExecutor
│   └── Many tasks → asyncio
│
├── Is it CPU bound?
│   ├── Has numpy-able loops? → NumPy vectorize
│   ├── Numerical loops?      → Numba @njit
│   ├── Multi-core needed?    → ProcessPoolExecutor
│   └── Need C speed?         → Cython
│
└── ML-specific?
    ├── Training slow?  → Mixed precision, gradient accum, DataLoader workers
    └── Inference slow? → torch.compile, quantization, batching
```

**Golden Rules:**
1. **Measure before optimizing.** Your intuition is wrong 80% of the time.
2. **Algorithm > code optimization.** A better algorithm beats micro-optimizations.
3. **Use built-ins.** They're implemented in C.
4. **Avoid Python loops over data.** Use NumPy/Pandas/built-ins.
5. **Memory is a bottleneck too.** Avoid copies, use generators, profile memory.
